# CN Analysis

In [ ]:
import scanpy as sc
import anndata as ad
import pandas as pd
import numpy as np
import os
import tifffile
import geopandas as gpd
from skimage import filters

import matplotlib.pyplot as plt
from matplotlib.pyplot import rc_context
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

### Aes

In [ ]:
labs_map = {
    "Adipocyte": "Adipocyte",
    "Bcell": "Bcell",
    "CD4_CCR6": "CD4 Tcell CCR6+",
    "CD4_GZMB": "CD4 Tcell GZMB+",
    "CD4_Other": "CD4 Tcell other",
    "CD4_Tex": "CD4 Tex",
    "CD4_Tmem": "CD4 Tmem",
    "CD4_Tn": "CD4 Tnaive",
    "CD4_Treg": "CD4 Treg",
    "CD8_GZMB": "CD8 Tcell GZMB+",
    "CD8_naive": "CD8 Tnaive",
    "CD8_Other": "CD8 Tcell other",
    "CD8_Tex": "CD8 Tex",
    "CD8_Tmem": "CD8 Tmem",
    "DC": "DC",
    "EC": "EC",
    "Lin-": "Unclassified",
    "Mac_M1": "M2-like macrophage",
    "Mac_M2": "M1-like macrophage",
    "MKs": "MK",
    "Mono_CD14": "Monocytes CD14+",
    "Mono_CD16": "Monocytes CD16+",
    "Mye_HLADR": "Myeloid HLA-DR+",
    "Mye_other": "Myeloid other",
    "Myeloma": "Myeloma",
    "NK": "NK cells"
}

color_palette_26_clean = {
    # CD4 (purples)
    "CD4 Tcell CCR6+": "#6A51A3",
    "CD4 Tcell GZMB+": "#9E9AC8",
    "CD4 Tcell other": "#807DBA",
    "CD4 Tex": "#BCBDDC",
    "CD4 Tmem": "#D5D3E6",
    "CD4 Tnaive": "#E6E4F1",
    "CD4 Treg": "#B39DD8",
    
    # CD8 (blues)
    "CD8 Tcell GZMB+": "#08519C",
    "CD8 Tcell other": "#2171B5",
    "CD8 Tex": "#6BAED6",
    "CD8 Tmem": "#9ECAE1",
    "CD8 Tnaive": "#C6DBEF",
    
    # Myeloid / APC (greens → yellow-greens)
    "DC": "#8C6BB1",
    "M1-like macrophage": "#005A32",
    "M2-like macrophage": "#238B45",
    "Monocytes CD14+": "#66C2A4",
    "Monocytes CD16+": "#A1D99B",
    "Myeloid HLA-DR+": "#C7E9B4",
    "Myeloid other": "#E5F5E0",
    
    # Other lineages (distinct hues)
    "Bcell": "#C49A6C",
    "EC": "#E5842B",
    "MK": "gold",
    "NK cells": "#D25C9E",
    "Adipocyte": "#F6E8C3",
    "Unclassified": "#6B6B6B",
    
    # Myeloma red
    "Myeloma": "#FF0000"
}

### Funs

In [ ]:
def img_map(img,figsize=(3,3),cmap='viridis',title=None, origin='lower',labels=False,cbar=False,save=None):
    fig, ax = plt.subplots(figsize=figsize)
    im = ax.imshow(img, cmap=cmap, origin=origin)
    ax.set_title(title)
    if not labels:
        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_xticklabels([])
        ax.set_yticklabels([])
    if cbar:
        plt.colorbar(im, ax=ax)
    if save is not None:
        plt.savefig(save, dpi=300, bbox_inches='tight', transparent=False) 
    plt.show()

def pctnorm(img, pct_lo = 1, pct_hi = 99.7):
    from skimage import exposure
    p_low, p_high = np.percentile(img, [pct_lo, pct_hi])
    return exposure.rescale_intensity(img, in_range=(p_low, p_high), out_range=(0, 1))

def mask2shapely(masks_in):
    from shapely.geometry import Polygon
    import geopandas as gpd
    from skimage import measure
    import numpy as np
    
    ids = []
    geometries = []

    for i in np.unique(masks_in)[1:]:
        binary_mask = (masks_in == i).astype(np.uint8)
        
        # Find contours using scikit-image
        contours = measure.find_contours(binary_mask, 0.5)
        
        # If multiple contours (disconnected pixels), take the longest one
        if len(contours) > 1:
            longest_contour = max(contours, key=len)
            # Convert from (row, col) to (x, y) coordinates
            coords = longest_contour[:, [1, 0]]
            if len(coords) > 3:
                geometries.append(Polygon(coords))
                ids.append(i)
        else:
            if len(contours) > 0 and len(contours[0]) > 3:
                # Convert from (row, col) to (x, y) coordinates
                coords = contours[0][:, [1, 0]]
                geometries.append(Polygon(coords))
                ids.append(i)
    
    gdf = gpd.GeoDataFrame({'id': ids, 'geometry': geometries}, crs='EPSG:4326')
    gdf = gdf[gdf['id'] != 0]
    gdf['x'] = gdf.geometry.centroid.x
    gdf['y'] = gdf.geometry.centroid.y   

    return gdf

def mask_map(img,mask, figsize=(4,4),fs=None, linewidth=0.75, title=None, origin='upper', cmap='viridis', labels=False):
    if fs is not None:
        figsize=fs


    gdf = mask2shapely(mask)
    
    fig,ax = plt.subplots(figsize=figsize)
    ax.imshow(img, cmap=cmap, interpolation='none', origin=origin)
    gdf.boundary.plot(aspect=1, ax=ax, color='white',linewidth=linewidth)
    ax.set_title(title)
    if not labels:
        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_xticklabels([])
        ax.set_yticklabels([])

In [ ]:
# Border graph functions

import os
import sys
from collections import defaultdict
import numpy as np
import pandas as pd
from scipy import ndimage
from scipy.sparse import csr_matrix,coo_matrix
from scipy.spatial import cKDTree
from skimage.segmentation import find_boundaries

# Calculate cell border lengths
def calculate_cell_border_lengths(mask):
    
    cell_ids = np.unique(mask)
    if 0 in cell_ids:
        cell_ids = cell_ids[cell_ids != 0]  # Remove background
    # sort IDs to ensure consistent ordering
    cell_ids = np.sort(cell_ids)
    n_cells = len(cell_ids)
    # mapping from cell ID to matrix index
    cell_to_index = {cell_id: idx for idx, cell_id in enumerate(cell_ids)}
    # dictionary to store border lengths between cell pairs (using matrix indices)
    border_lengths = defaultdict(float) 
    # horizontal boundaries (between horizontally adjacent pixels)
    for i in range(mask.shape[0]):
        for j in range(mask.shape[1] - 1):
            cell1 = mask[i, j]
            cell2 = mask[i, j + 1]
            # only count if both cells non-background and different
            if cell1 != 0 and cell2 != 0 and cell1 != cell2:
                # convert cell IDs to matrix indices
                idx1 = cell_to_index[cell1]
                idx2 = cell_to_index[cell2]
                # use tuple with smaller index first for consistency
                pair = (min(idx1, idx2), max(idx1, idx2))
                border_lengths[pair] += 1.0  # Each pixel boundary has length 1
    # vertical boundaries (between vertically adjacent pixels)
    for i in range(mask.shape[0] - 1):
        for j in range(mask.shape[1]):
            cell1 = mask[i, j]
            cell2 = mask[i + 1, j] 
            # only count if both cells are non-background and different
            if cell1 != 0 and cell2 != 0 and cell1 != cell2:
                # convert cell IDs to matrix indices
                idx1 = cell_to_index[cell1]
                idx2 = cell_to_index[cell2]
                # use tuple with smaller index first for consistency
                pair = (min(idx1, idx2), max(idx1, idx2))
                border_lengths[pair] += 1.0  # Each pixel boundary has length 1
    # convert to sparse matrix format
    if not border_lengths:
        # No borders found, return empty sparse matrix
        return coo_matrix((n_cells, n_cells))
    # create coordinate arrays for sparse matrix
    rows = []
    cols = []
    data = []
    for (idx1, idx2), length in border_lengths.items():
        # Add both (i,j) and (j,i) entries to make the matrix symmetric
        rows.extend([idx1, idx2])
        cols.extend([idx2, idx1])
        data.extend([length, length])
    # Create sparse matrix with dimensions N x N where N is number of cells
    border_matrix = coo_matrix((data, (rows, cols)), shape=(n_cells, n_cells))
    return border_matrix,cell_ids

def calculate_cell_border_lengths_dilated(mask, distance=1):
    cell_ids = np.unique(mask)
    if 0 in cell_ids:
        cell_ids = cell_ids[cell_ids != 0]
    cell_ids = np.sort(cell_ids)
    n_cells = len(cell_ids)
    cell_to_index = {cell_id: idx for idx, cell_id in enumerate(cell_ids)}
    
    border_lengths = defaultdict(float)
    
    # Scan horizontal boundaries within distance
    for i in range(mask.shape[0]):
        for j in range(mask.shape[1] - distance):
            cell1 = mask[i, j]
            cell2 = mask[i, j + distance]
            if cell1 != 0 and cell2 != 0 and cell1 != cell2:
                idx1 = cell_to_index[cell1]
                idx2 = cell_to_index[cell2]
                pair = (min(idx1, idx2), max(idx1, idx2))
                border_lengths[pair] += 1.0
    
    # Scan vertical boundaries within distance
    for i in range(mask.shape[0] - distance):
        for j in range(mask.shape[1]):
            cell1 = mask[i, j]
            cell2 = mask[i + distance, j]
            if cell1 != 0 and cell2 != 0 and cell1 != cell2:
                idx1 = cell_to_index[cell1]
                idx2 = cell_to_index[cell2]
                pair = (min(idx1, idx2), max(idx1, idx2))
                border_lengths[pair] += 1.0
    
    if not border_lengths:
        return coo_matrix((n_cells, n_cells)), cell_ids
    
    rows, cols, data = [], [], []
    for (idx1, idx2), length in border_lengths.items():
        rows.extend([idx1, idx2])
        cols.extend([idx2, idx1])
        data.extend([length, length])
    
    border_matrix = coo_matrix((data, (rows, cols)), shape=(n_cells, n_cells))
    return border_matrix, cell_ids

In [ ]:

import numpy as np
import pandas as pd
from sklearn.cluster import MiniBatchKMeans
from scipy import sparse

def compute_neighbor_composition(connectivity, labels):

    labels = np.asarray(labels)
    celltypes = np.unique(labels)
    n_cells = len(labels)
    n_types = len(celltypes)
    
    # Map labels to indices
    label_to_idx = {ct: i for i, ct in enumerate(celltypes)}
    label_indices = np.array([label_to_idx[l] for l in labels])
    
    # Convert to binary matrices for each cell type
    composition = np.zeros((n_cells, n_types), dtype=np.float32)
    
    for i, ct in enumerate(celltypes):
        # Get cells of this type
        is_type = (labels == ct).astype(float)
        # Sum neighbors of this type for each cell
        neighbor_counts = connectivity.dot(is_type)
        composition[:, i] = neighbor_counts
    
    # Normalize to proportions
    totals = composition.sum(axis=1, keepdims=True)
    composition = np.divide(composition, totals, 
                           where=totals>0, 
                           out=composition)
    
    return composition, celltypes


def cluster_neighborhoods(composition, n_clusters=10, random_state=0):

    model = MiniBatchKMeans(
        n_clusters=n_clusters,
        random_state=random_state,
        batch_size=1000,
        n_init=10
    )
    clusters = model.fit_predict(composition)
    return clusters, model


def analyze_sample(connectivity, labels, n_clusters=10, sample_id=None):

    composition, celltypes = compute_neighbor_composition(connectivity, labels)
    cn_labels, model = cluster_neighborhoods(composition, n_clusters)
    
    return {
        'cn_labels': cn_labels,
        'composition': composition,
        'celltypes': celltypes,
        'model': model,
        'sample_id': sample_id
    }


def analyze_multiple_samples(connectivity_dict, labels_dict, n_clusters=10):

    print(f"Computing neighborhood composition for {len(connectivity_dict)} samples...")
    
    # First pass: get union of all cell types
    all_celltypes = set()
    for sample_id in connectivity_dict:
        all_celltypes.update(np.unique(labels_dict[sample_id]))
    all_celltypes = sorted(all_celltypes)
    n_types = len(all_celltypes)
    celltype_to_idx = {ct: i for i, ct in enumerate(all_celltypes)}
    print(f"Found {n_types} unique cell types across all samples")
    
    # Second pass: compute aligned compositions
    compositions = []
    sample_ids = []
    cell_counts = []
    
    for sample_id in sorted(connectivity_dict.keys()):
        # Get raw composition
        comp_raw, sample_celltypes = compute_neighbor_composition(
            connectivity_dict[sample_id],
            labels_dict[sample_id]
        )
        
        # Align to unified cell type order
        comp_aligned = np.zeros((len(comp_raw), n_types), dtype=np.float32)
        for i, ct in enumerate(sample_celltypes):
            unified_idx = celltype_to_idx[ct]
            comp_aligned[:, unified_idx] = comp_raw[:, i]
        
        compositions.append(comp_aligned)
        sample_ids.append(sample_id)
        cell_counts.append(len(comp_aligned))
        print(f"  {sample_id}: {len(comp_aligned)} cells, {len(sample_celltypes)} cell types")
    
    # Stack all compositions together
    all_composition = np.vstack(compositions)
    print(f"\nClustering {all_composition.shape[0]} total cells into {n_clusters} CNs...")
    
    # Single unified clustering
    cn_labels, model = cluster_neighborhoods(all_composition, n_clusters)
    
    # Split results back by sample
    results = {}
    start_idx = 0
    for i, sample_id in enumerate(sample_ids):
        end_idx = start_idx + cell_counts[i]
        
        results[sample_id] = {
            'cn_labels': cn_labels[start_idx:end_idx],
            'composition': compositions[i],
            'celltypes': all_celltypes,  # Use unified cell types
            'model': model,
            'sample_id': sample_id
        }
        
        start_idx = end_idx
    
    return results, model


def summarize_cns(results, labels_dict):

    summaries = []
    
    for sample_id, res in results.items():
        composition = res['composition']
        cn_labels = res['cn_labels']
        celltypes = res['celltypes']
        
        # Mean composition per CN
        df = pd.DataFrame(composition, columns=celltypes)
        df['CN'] = cn_labels
        df['sample'] = sample_id
        
        summary = df.groupby(['sample', 'CN']).mean().reset_index()
        summaries.append(summary)
    
    return pd.concat(summaries, ignore_index=True)



### Calculate cell border lengths (contact distances) (_run once_)

In [ ]:
obs_data = pd.read_csv('/mnt/disks/data/imc/CART_cohort/out/v1_pheno_work/obs_v5.csv')
obs_data.head(1)

In [ ]:
md = pd.read_csv("/mnt/disks/data/imc/CART_cohort/in/sample_md.csv")
from tqdm import tqdm

for sample_id in tqdm(md['sample'].unique()):

    if len(os.listdir(f"/mnt/disks/data/imc/CART_cohort/in/sample_gating/{sample_id}"))==0:
        continue
    if 'mask_V1withAdipo_MKv2.tif' not in os.listdir(f'/mnt/disks/data/imc/CART_cohort/out/sample_data/{sample_id}/segmentation_masks/'):
        continue 

    #import
    mask = tifffile.imread(f'/mnt/disks/data/imc/CART_cohort/out/sample_data/{sample_id}/segmentation_masks/mask_V1withAdipo_MKv2.tif')
    obs_sample = obs_data[obs_data.sample_id==sample_id]
    obs_sample = obs_sample.set_index('Cell_ID')
    
    #GATE CELLS
    obs_sample = obs_sample[obs_sample.gated]
    
    #overlapping cells, index
    shared_cells = np.intersect1d(mask,obs_sample.index)
    mask = np.where(np.isin(mask,shared_cells),mask,0)
    obs_sample = obs_sample.loc[shared_cells,]
    
    # Calc cell border length
    border_lengths,cell_ids = calculate_cell_border_lengths(mask)
    
    #save results
    from scipy.sparse import save_npz
    save_npz(f"/mnt/disks/data/imc/CART_cohort/out/sample_data/{sample_id}/segmentation_masks/mask_V1withAdipo_MKv2-border_lengths.npz", 
             border_lengths) 
    pd.DataFrame({"Cell_IDs":cell_ids}).to_csv(
        f"/mnt/disks/data/imc/CART_cohort/out/sample_data/{sample_id}/segmentation_masks/mask_V1withAdipo_MKv2-cell_ids.csv",index=False
    )

In [ ]:
# Also run on DILATED border, 10um as per original

dist_10um_pixels = int(1440/2000*10)

md = pd.read_csv("/mnt/disks/data/imc/CART_cohort/in/sample_md.csv")
from tqdm import tqdm

for sample_id in tqdm(md['sample'].unique()):

    if len(os.listdir(f"/mnt/disks/data/imc/CART_cohort/in/sample_gating/{sample_id}"))==0:
        continue
    if 'mask_V1withAdipo_MKv2.tif' not in os.listdir(f'/mnt/disks/data/imc/CART_cohort/out/sample_data/{sample_id}/segmentation_masks/'):
        continue 

    #import
    mask = tifffile.imread(f'/mnt/disks/data/imc/CART_cohort/out/sample_data/{sample_id}/segmentation_masks/mask_V1withAdipo_MKv2.tif')
    obs_sample = obs_data[obs_data.sample_id==sample_id]
    obs_sample = obs_sample.set_index('Cell_ID')
    
    #GATE CELLS
    obs_sample = obs_sample[obs_sample.gated]
    
    #overlapping cells, index
    shared_cells = np.intersect1d(mask,obs_sample.index)
    mask = np.where(np.isin(mask,shared_cells),mask,0)

    # Calc cell border length - dilated distance
    border_lengths = calculate_cell_border_lengths_dilated(mask, dist_10um_pixels )[0]
    
    #save results
    from scipy.sparse import save_npz
    save_npz(
        f"/mnt/disks/data/imc/CART_cohort/out/sample_data/{sample_id}/segmentation_masks/mask_V1withAdipo_MKv2-border_lengths-dilated.npz", 
        border_lengths) 


### Run iterations

In [ ]:
def calc_pct_table(df_in, clus):
    
    clus_ids = df_in[clus].unique().tolist()
    
    df = df_in[['sample_id',clus]].value_counts().reset_index()\
        .merge(df_in[['sample_id']].value_counts().reset_index().rename({'count':'total'},axis=1))
    
    df = df.merge(
        pd.read_csv('/mnt/disks/data/imc/CART_cohort/in/sample_id-clinical.csv')
    ).drop('sample_id',axis=1).groupby(['slideID','Timepoint',clus]).sum().reset_index()
    
    df['pct'] = df['count']/df['total']
    df = df.pivot(index=['slideID','Timepoint'],columns=clus, values='pct').fillna(0).reset_index()
    
    df = df[ df[clus_ids].sum(axis=1)>0 ]

    df[clus_ids] = df[clus_ids]*100
    
    return df

#### Import

In [ ]:
from scipy.sparse import load_npz

md = pd.read_csv("/mnt/disks/data/imc/CART_cohort/in/sample_md.csv")

obs_data = pd.read_csv('/mnt/disks/data/imc/CART_cohort/out/v1_pheno_work/obs_v5.csv')
clus = 'ph'
clus_remove = ['Lin-','B-T','Artifact']

connectivity_dict = {}
labels_dict = {}
obs_dict = {}

for sample_id in md['sample'].unique():
    
    if len(os.listdir(f"/mnt/disks/data/imc/CART_cohort/in/sample_gating/{sample_id}"))==0:
        continue
    if 'mask_V1withAdipo_MKv2.tif' not in os.listdir(f'/mnt/disks/data/imc/CART_cohort/out/sample_data/{sample_id}/segmentation_masks/'):
        continue 
    if sample_id == "BS-23-N56046_002":
        continue

    obs_sample = obs_data[obs_data.sample_id==sample_id]
    obs_sample = obs_sample.set_index('Cell_ID')
    
    loc_ = f'/mnt/disks/data/imc/CART_cohort/out/sample_data/{sample_id}/segmentation_masks/'
    border_lengths = load_npz(loc_+'mask_V1withAdipo_MKv2-border_lengths-dilated.npz')
    cell_ids = pd.read_csv(loc_+'mask_V1withAdipo_MKv2-cell_ids.csv')['Cell_IDs'].tolist()
    
    #filter obs by ph
    obs_sample = obs_sample[~obs_sample[clus].isin(clus_remove)]
    #intersection of cell IDs
    cells_use = np.intersect1d(obs_sample.index, cell_ids)
    #integer indices for matrix slicing
    idx = np.searchsorted(cell_ids, cells_use)
    #slice matrix and obs
    border_lengths = border_lengths.tocsr()[idx, :][:, idx].tocoo()
    obs_sample = obs_sample.loc[cells_use]
    
    labels = obs_sample[clus].tolist()
    
    connectivity_dict[sample_id] = border_lengths
    labels_dict[sample_id] = labels
    obs_dict[sample_id] = obs_sample.copy()

#### Cluster

In [ ]:
md = pd.read_csv('/mnt/disks/data/imc/CART_cohort/in/sample_id-clinical.csv') 
n_cluster_options = [10,12,15]
saveloc = "/mnt/disks/data/imc/CART_cohort/out/v1_pheno_work/cn_data/"

In [ ]:
# All TP, no Lin-

run_ = 'allTP_linNegOut'

for n_clusters in n_cluster_options:
    
    results = analyze_multiple_samples(connectivity_dict=connectivity_dict,labels_dict=labels_dict,n_clusters=n_clusters)
    
    for sample_id in results[0].keys():
        obs_dict[sample_id]['cn_labels'] = ['CN'+str(i) for i in results[0][sample_id]['cn_labels'].tolist()]
    
    df = pd.concat(obs_dict.values(),axis=0).reset_index()
    #df.to_csv(f"{saveloc}{run_}_{str(n_clusters)}clus-cells_data.csv",index=False)
    #calc_pct_table(df, "cn_labels").to_csv(f"{saveloc}{run_}_{str(n_clusters)}clus-sample_pct.csv",index=False)
    break

In [ ]:
# bsl TP, no Lin-

run_ = 'bslTP_linNegOut'

samples_sub = md[md.Timepoint=='Baseline'].sample_id.tolist()

#subset data
connectivity_dict_sub = {}
labels_dict_sub = {}
obs_dict_sub = {}
for sample_id in samples_sub:
    if sample_id not in connectivity_dict.keys():
        continue
    connectivity_dict_sub[sample_id] = connectivity_dict[sample_id]
    labels_dict_sub[sample_id] = labels_dict[sample_id]
    obs_dict_sub[sample_id] = obs_dict[sample_id]

#run/save
for n_clusters in n_cluster_options:
    
    results = analyze_multiple_samples(connectivity_dict=connectivity_dict_sub,labels_dict=labels_dict_sub,n_clusters=n_clusters)
    
    for sample_id in results[0].keys():
        obs_dict_sub[sample_id]['cn_labels'] = ['CN'+str(i) for i in results[0][sample_id]['cn_labels'].tolist()]
    
    df = pd.concat(obs_dict_sub.values(),axis=0).reset_index()
    df.to_csv(f"{saveloc}{run_}_{str(n_clusters)}clus-cells_data.csv",index=False)
    calc_pct_table(df, "cn_labels").to_csv(f"{saveloc}{run_}_{str(n_clusters)}clus-sample_pct.csv",index=False)


In [ ]:
# post TP, no Lin-

run_ = 'postTP_linNegOut'

samples_sub = md[md.Timepoint=='01MOpost'].sample_id.tolist()

#subset data
connectivity_dict_sub = {}
labels_dict_sub = {}
obs_dict_sub = {}
for sample_id in samples_sub:
    if sample_id not in connectivity_dict.keys():
        continue
    connectivity_dict_sub[sample_id] = connectivity_dict[sample_id]
    labels_dict_sub[sample_id] = labels_dict[sample_id]
    obs_dict_sub[sample_id] = obs_dict[sample_id]

#run/save
for n_clusters in n_cluster_options:
    
    results = analyze_multiple_samples(connectivity_dict=connectivity_dict_sub,labels_dict=labels_dict_sub,n_clusters=n_clusters)
    
    for sample_id in results[0].keys():
        obs_dict_sub[sample_id]['cn_labels'] = ['CN'+str(i) for i in results[0][sample_id]['cn_labels'].tolist()]
    
    df = pd.concat(obs_dict_sub.values(),axis=0).reset_index()
    df.to_csv(f"{saveloc}{run_}_{str(n_clusters)}clus-cells_data.csv",index=False)
    calc_pct_table(df, "cn_labels").to_csv(f"{saveloc}{run_}_{str(n_clusters)}clus-sample_pct.csv",index=False)


### EMD CN

#### Borders Calculation

In [ ]:
obs_data = sc.read('/mnt/disks/data/imc/CART_cohort/out/emd_work/emd_clustering-all.h5ad').obs

In [ ]:
# DILATED border, 10um as per original

dist_10um_pixels = int(1440/2000*10)

sample_ids = [i.split('.')[0] for i in os.listdir(f"/mnt/disks/data/imc/CART_cohort/in/emd_tiffs/masks/")]

for sample_id in sample_ids:

    #import
    mask = tifffile.imread(f"/mnt/disks/data/imc/CART_cohort/in/emd_tiffs/masks/{sample_id}.tiff")
    obs_sample = obs_data[obs_data.sample_id==sample_id]
    obs_sample = obs_sample.set_index('ObjectNumber')
    
    #overlapping cells, index
    shared_cells = np.intersect1d(mask,obs_sample.index)
    mask = np.where(np.isin(mask,shared_cells),mask,0)

    # Calc cell border length - dilated distance
    border_lengths,cell_ids = calculate_cell_border_lengths_dilated(mask, dist_10um_pixels )
    
    #save results
    from scipy.sparse import save_npz
    save_npz(
        f"/mnt/disks/data/imc/CART_cohort/out/emd_data/{sample_id}-border_lengths_dilated.npz", 
        border_lengths) 
    pd.DataFrame({"Cell_IDs":cell_ids}).to_csv(
        f"/mnt/disks/data/imc/CART_cohort/out/emd_data/{sample_id}-border_lengths_cellids.npz",index=False
    )


#### Run CNs

In [ ]:
# Import

from scipy.sparse import load_npz

sample_ids = [i.split('.')[0] for i in os.listdir(f"/mnt/disks/data/imc/CART_cohort/in/emd_tiffs/masks/")]

obs_data = sc.read('/mnt/disks/data/imc/CART_cohort/out/emd_work/emd_clustering-all.h5ad').obs
obs_data['lineage_2'] = np.where(
    (obs_data['lineage'] == 'PC') & (obs_data['Ki67_pos'] == True),
    'PC_Ki67',
    obs_data['lineage']
)
clus = 'lineage_2'
clus_remove = ['Unclassified']

connectivity_dict = {}
labels_dict = {}
obs_dict = {}

for sample_id in sample_ids:

    obs_sample = obs_data[obs_data.sample_id==sample_id]
    obs_sample = obs_sample.set_index('ObjectNumber')
    
    border_lengths = load_npz(f"/mnt/disks/data/imc/CART_cohort/out/emd_data/{sample_id}-border_lengths_dilated.npz")
    cell_ids = pd.read_csv(f"/mnt/disks/data/imc/CART_cohort/out/emd_data/{sample_id}-border_lengths_cellids.npz")['Cell_IDs'].tolist()

    #subset
    obs_sample = obs_sample[~obs_sample[clus].isin(clus_remove)]
    
    #intersection of cell IDs
    cells_use = np.intersect1d(obs_sample.index, cell_ids)
    #integer indices for matrix slicing
    idx = np.searchsorted(cell_ids, cells_use)
    #slice matrix and obs
    border_lengths = border_lengths.tocsr()[idx, :][:, idx].tocoo()
    obs_sample = obs_sample.loc[cells_use]
    
    labels = obs_sample[clus].tolist()
    
    connectivity_dict[sample_id] = border_lengths
    labels_dict[sample_id] = labels
    obs_dict[sample_id] = obs_sample.copy()

In [ ]:
# Run over

n_cluster_options = [5]
saveloc = "/mnt/disks/data/imc/CART_cohort/out/v1_pheno_work/cn_data/"
run_ = 'EMD'

for n_clusters in n_cluster_options:
    
    results = analyze_multiple_samples(connectivity_dict=connectivity_dict,labels_dict=labels_dict,n_clusters=n_clusters)
    
    for sample_id in results[0].keys():
        obs_dict[sample_id]['cn_labels'] = ['CN'+str(i) for i in results[0][sample_id]['cn_labels'].tolist()]
    
    df = pd.concat(obs_dict.values(),axis=0).reset_index()
    df.to_csv(f"{saveloc}{run_}_{str(n_clusters)}clus-cells_data.csv",index=False)

    df_in=df
    clus="cn_labels"
    
    clus_ids = df_in[clus].unique().tolist()
    
    df_in = df_in[['sample_id',clus]].value_counts().reset_index()\
        .merge(df_in[['sample_id']].value_counts().reset_index().rename({'count':'total'},axis=1))
       
    df_in['pct'] = df_in['count']/df_in['total']
    df_in = df_in.pivot(index=['sample_id'],columns=clus, values='pct').fillna(0).reset_index()
    df_in[clus_ids] = df_in[clus_ids]*100
    df_in.to_csv(f"{saveloc}{run_}_{str(n_clusters)}clus-sample_pct.csv",index=False)